# Transforms, decomposition and factor geometry

Family 4 turns coordinate maps, differences, periodic bases and factor geometry into independent mathematical objects. Everything here is generated from equations. No trading application, calendar or data provider is involved.

We work through scaling, fitted composition, PCA, whitening, lag alignment, seasonal differences and their inverses, deterministic features, Gaussian factors, projections and subspace alignment. Every example checks a mathematical identity. [Catalog and contracts](../docs/transforms.md).

Run with an environment installed using `python -m pip install -e '.[notebooks]'`.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "convexity" / "math").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose
from convexity.math import transforms as tx

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "grid.alpha": .2})

## 1. Fit a coordinate map, then reuse it

For row vectors, standardization is $z=(x-\mu)/s$. The parameters belong to the supplied training matrix. Applying the map to later rows never changes them. Scaling is a mathematical choice: PCA of a covariance matrix and PCA of a correlation matrix solve different problems.

Here a two-dimensional latent signal generates five noisy observations. We split the generated rows simply to demonstrate frozen fitted state.

In [ ]:
B = np.array([[2., 0.], [1., .2], [0., 2.], [.2, 1.5], [1., 1.]])
psi = np.full(5, .35**2)
latent = rng.normal(size=(2400, 2))
x = latent @ B.T + rng.normal(size=(2400, 5)) * np.sqrt(psi)
training, held_out = x[:1800], x[1800:]
scaling = tx.standardize(training)
assert_allclose(scaling.transform(training).mean(axis=0), 0., atol=1e-14)
assert_allclose(scaling.transform(training).var(axis=0), 1., atol=1e-14)
assert_allclose(scaling.inverse_transform(scaling.transform(held_out)), held_out, atol=1e-14)
pd.DataFrame({"training location": scaling.location, "training scale": scaling.scale})

In [ ]:
contaminated = np.r_[rng.normal(size=(100, 1)), [[40.]]]
ordinary = tx.standardize(contaminated)
robust = tx.robust_scale(contaminated)
ranged = tx.minmax_scale(contaminated, lower=-1, upper=1)
assert_allclose(ranged.inverse_transform(ranged.transform(contaminated)), contaminated, atol=1e-13)
pd.DataFrame({"standard": [ordinary.location[0], ordinary.scale[0]],
              "robust": [robust.location[0], robust.scale[0]]}, index=["location", "scale"])

## 2. Principal coordinates and omitted directions

For the centered training matrix $X_c=U S Q^\top$, retaining the first $k$ columns of $Q$ gives scores $Z=X_c Q_k$. Reconstruction is $ZQ_k^\top+\mu$. The residual is orthogonal to the retained basis, and its squared Frobenius norm equals the sum of omitted squared singular values.

`components` stores axes **in columns**. `ddof=1` means each explained variance is $s_j^2/(n-1)$. Reduced reconstruction has no exact inverse over the original five-dimensional space.

In [ ]:
principal = tx.pca(training, n_components=2)
scores = principal.transform(training)
residuals = principal.residuals(training)
singular = np.linalg.svd(training - training.mean(axis=0), compute_uv=False)
assert_allclose(residuals @ principal.components, 0., atol=1e-12)
assert_allclose(np.sum(residuals**2), np.sum(singular[2:]**2))
assert_allclose(principal.reconstruct(scores) + residuals, training)
assert not principal.is_invertible

full = tx.pca(training)
assert_allclose(full.inverse_transform(full.transform(held_out)), held_out, atol=1e-12)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(np.arange(1, 6), full.explained_variance_ratio)
axes[0].set(xlabel="principal component", ylabel="fraction of total variance", title="Two dominant directions")
axes[1].scatter(scores[:, 0], scores[:, 1], s=5, alpha=.25)
axes[1].set(xlabel="PC 1", ylabel="PC 2", title="Principal coordinates")
plt.tight_layout()
plt.show()

## 3. Composition preserves the meaning of each stage

A pipeline fits each stage on the previous stage's transformed training rows. Its residual is measured back in the original observation units. A full-rank pipeline can invert batches such as `(path, step, coordinate)` arrays.

In [ ]:
pipeline = tx.Pipeline((tx.Standardize(), tx.PCA(2))).fit(training)
held_scores = pipeline.transform(held_out)
approximation = pipeline.reconstruct(held_scores)
assert_allclose(approximation + pipeline.residuals(held_out), held_out)
exact = tx.Pipeline((tx.RobustScale(), tx.PCA())).fit(training)
batch = held_out[:60].reshape(10, 6, 5)
assert_allclose(exact.inverse_transform(exact.transform(batch)), batch, atol=1e-12)
print("Held-out reconstruction RMSE:", np.sqrt(np.mean((held_out - approximation)**2)))
print("Batch shape through exact inverse:", batch.shape)

## 4. Whitening is an explicit variance normalization

PCA whitening uses $W=Q\Lambda^{-1/2}$. ZCA uses $W=Q\Lambda^{-1/2}Q^\top$, keeping the original coordinate axes. On the training matrix, covariance with the declared divisor is identity. A held-out covariance need not equal identity.

Zero-variance directions cannot be whitened. They must be omitted explicitly; this API never silently adds a ridge or clips eigenvalues.

In [ ]:
cloud = rng.normal(size=(900, 2)) @ np.array([[3., 1.5], [0., .5]]) + [2., -1.]
pca_white = tx.whitening(cloud, method="pca")
zca_white = tx.whitening(cloud, method="zca")
for mapping in (pca_white, zca_white):
    assert_allclose(np.cov(mapping.transform(cloud).T), np.eye(2), atol=1e-12)
    assert_allclose(mapping.inverse_transform(mapping.transform(cloud)), cloud, atol=1e-12)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, values, title in zip(axes, [cloud, pca_white.transform(cloud), zca_white.transform(cloud)],
                           ["Original cloud", "PCA whitening", "ZCA whitening"]):
    ax.scatter(values[:, 0], values[:, 1], s=5, alpha=.25)
    ax.set(title=title, xlabel="coordinate 1", ylabel="coordinate 2")
    ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()

rank_one = np.column_stack((np.arange(10.), 2*np.arange(10.)))
try:
    tx.whitening(rank_one)
except ValueError as error:
    print("Explicit rank failure:", error)
reduced = tx.whitening(rank_one, method="pca", n_components=1)
assert_allclose(reduced.transform(rank_one).var(ddof=1), 1.)

## 5. Differences need boundary conditions to be inverted

The seasonal difference is $D_s x_t=x_t-x_{t-s}$. Higher order means repeated application, $D_s^q$. Dropping the first $sq$ rows loses boundary information unless we retain it.

`DifferenceResult` stores leading and trailing blocks at every difference level. Ordinary inversion restores the original sequence, including its leading rows. Continuation integrates supplied future differences from the trailing blocks and returns only future rows. This is integration, not prediction.

In [ ]:
t = np.arange(100.)
y = .01*t**2 + np.sin(2*np.pi*t/12)
training_y = y[:70]
delta = tx.difference(training_y, lag=12, order=2)
assert_allclose(delta.inverse_transform(), training_y, atol=1e-12)
whole_delta = tx.difference(y, lag=12, order=2)
future_delta = whole_delta.values[70 - delta.row_offset:]
restored_future = delta.inverse_transform(future_delta, continuation=True)
assert_allclose(restored_future, y[70:], atol=1e-12)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(t[:70], training_y, label="supplied training sequence")
axes[0].plot(t[70:], restored_future, "--", label="integrated future differences")
axes[0].set(xlabel="position", ylabel="level", title="Boundary-aware integration")
axes[0].legend()
axes[1].plot(delta.rows, delta.values)
axes[1].set(xlabel="original row position", ylabel="difference", title="Second seasonal difference")
plt.tight_layout()
plt.show()

## 6. Lagged designs expose their row alignment

Lag zero is the current observation. All positive lags refer to preceding positions. The feature matrix retains only complete rows and labels its column ordering. Alignment is positional and independent of any calendar.

In [ ]:
lagged = tx.lag_features(y, lags=(0, 1, 12))
assert_allclose(lagged.values[:, 0], y[lagged.rows])
assert_allclose(lagged.values[:, 2], y[lagged.rows - 12])
pd.DataFrame(lagged.values[:6], index=lagged.rows[:6], columns=lagged.columns)

## 7. Deterministic periodic bases continue their phase

Fourier columns are interleaved sine/cosine pairs. Seasonal columns are indicator vectors for `position % period`. An explicit `start` continues a design without resetting phase. An intercept is never inserted, and seasonal baseline removal is explicit.

In [ ]:
terms = tx.FourierFeatures(period=12, order=3)
a, b, full_terms = terms(18), terms(18, start=18), terms(36)
assert_allclose(np.vstack((a.values, b.values)), full_terms.values, atol=1e-14)
assert_allclose(full_terms.values.T @ full_terms.values, 18*np.eye(6), atol=1e-12)
seasonal = tx.seasonal_features(36, period=12, drop_first=True)
assert seasonal.values.shape == (36, 11)

fig, ax = plt.subplots()
for j in range(4):
    ax.plot(full_terms.rows, full_terms.values[:, j], label=full_terms.columns[j])
ax.axvline(18, color="black", linestyle=":", label="continuation boundary")
ax.set(xlabel="position", ylabel="basis value", title="Fourier phase survives a split")
ax.legend(ncol=3)
plt.show()

## 8. A Gaussian factor model is more than a projection

Let $x=\mu+Bf+\epsilon$, with $f\sim N(0,I)$ and independent $\epsilon\sim N(0,\Psi)$, where $\Psi$ is diagonal and positive. The implied observation covariance is $BB^\top+\Psi$.

The posterior factor covariance is $V=(I+B^\top\Psi^{-1}B)^{-1}$. Scores are posterior means $E[f\mid x]=V B^\top\Psi^{-1}(x-\mu)$, expressed as row vectors by the API. Reconstructing those means gives the common component. Its remainder is generally not an orthogonal residual.

In [ ]:
known = tx.GaussianFactorModel(np.zeros(5), B, psi)
conditional_encoder = np.linalg.solve(known.covariance, B)
assert_allclose(known.transform(held_out), held_out @ conditional_encoder, atol=1e-12)
assert_allclose(known.posterior_covariance, np.eye(2) - B.T @ conditional_encoder, atol=1e-12)

factors = tx.factor_analysis(training, n_components=2, rotation="varimax")
assert factors.diagnostics.converged
factor_scores = factors.transform(held_out)
common = factors.reconstruct(factor_scores)
assert_allclose(common + factors.residuals(held_out), held_out)
print(factors.diagnostics)
pd.DataFrame({"generating noise variance": psi, "fitted noise variance": factors.noise_variance})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, covariance, title in zip(axes, [known.covariance, factors.covariance],
                                ["Generating covariance", "Fitted factor covariance"]):
    im = ax.imshow(covariance, cmap="viridis", vmin=0, vmax=5)
    ax.set(title=title, xlabel="coordinate", ylabel="coordinate")
    ax.grid(False)
fig.colorbar(im, ax=axes, label="covariance", shrink=.85)
plt.show()

## 9. Subspaces, residuals and alignment

For orthonormal columns $Q$, the projector is $P=QQ^\top$. A change of basis $Q\mapsto QR$ with orthogonal $R$ preserves the subspace. Rotating row scores by the same $R$ preserves reconstruction.

Principal angles compare the spaces themselves. Orthogonal Procrustes aligns basis choices, including signs, permutations and rotations, but cannot remove a difference between the spanned spaces. Factors inside tied eigenspaces are not uniquely identified.

In [ ]:
space = tx.subspace(B)
projection, residual = space.project(held_out), space.residuals(held_out)
assert_allclose(projection + residual, held_out)
assert_allclose(residual @ space.basis, 0., atol=1e-12)
assert_allclose(np.sum(held_out**2), np.sum(projection**2) + np.sum(residual**2))

angle = .8
R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
rotated = tx.Subspace(space.basis @ R)
alignment = tx.align_subspaces(space, rotated)
z = space.coordinates(held_out)
assert_allclose(alignment.aligned.reconstruct(alignment.transform(z)), projection, atol=1e-12)
assert_allclose(alignment.angles, 0., atol=1e-12)
print("Basis alignment error:", alignment.error)
print("Generating / fitted factor subspace angles (degrees):",
      np.degrees(tx.principal_angles(space, tx.subspace(factors.loadings))))

## 10. Supply a mathematical map of your own

A fitter needs only a `.fit(samples)` method that returns a checked coordinate map. There is no subclass, backend registry or application object to implement. `fit` and `Pipeline` verify the dimensional contract.

In [ ]:
class Center:
    def fit(self, samples):
        identity = np.eye(samples.shape[1])
        return tx.AffineTransform(samples.mean(axis=0), identity, identity)

custom = tx.fit(training, Center())
assert_allclose(custom.transform(training).mean(axis=0), 0., atol=1e-14)
custom_pipeline = tx.Pipeline((Center(), tx.PCA(2))).fit(training)
assert custom_pipeline.transform(held_out).shape == (len(held_out), 2)
print("All mathematical identities in this notebook passed.")